# Q4/Q5/Q6 — Hyperparameter Sweep & Analysis

In [ ]:
import sys
sys.path.append('..')
import yaml

## Q4 — Sweep Strategy: Bayesian Optimization

Grid search = 2×3×3×3×2×6×3×2×3 = **11,664 runs** — not feasible.

We used **Bayesian search** because:
1. It builds a probabilistic model of the objective function
2. It picks hyperparameter combos that are likely to improve on past results
3. Converges to good configs much faster than random search
4. ~50-100 runs covers the space well enough

In [ ]:
with open('../sweep.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
print('Sweep parameters:')
for k, v in cfg['parameters'].items():
    print(f'  {k}: {v}')

In [ ]:
# to run the sweep:
# wandb sweep sweep.yaml
# wandb agent <sweep_id> --count 50

## Q5 — Best Validation Accuracy

*Add wandb panel here*

Best validation accuracy: **89.00%** with:
- Optimizer: Adam
- Hidden layers: 3 × 128
- Activation: ReLU
- Weight init: Xavier
- Learning rate: 0.001
- Batch size: 32
- Weight decay: 0
- Epochs: 15

## Q6 — Observations

*Add parallel coordinates plot and correlation summary from wandb here*

### Key observations:

1. **Adam and NAdam dominate** — they appear in all top-10 runs. Adaptive learning rates handle varying gradient magnitudes much better than vanilla SGD.

2. **ReLU >> sigmoid/tanh** — sigmoid plateaus around 80-82% due to vanishing gradients in 3+ layer networks. tanh is slightly better (~83-85%) but still worse than ReLU (~88-89%).

3. **Xavier init is critical** — random init leads to unstable training with sigmoid/tanh. With ReLU it matters less but Xavier still converges faster.

4. **Weight decay 0.5 kills everything** — accuracy drops below 60% regardless of other hyperparameters. The regularization is too aggressive. 0.0005 gives mild benefit, 0 is fine.

5. **Batch size 32 is the sweet spot** — 16 is too noisy (high variance in loss), 64 converges faster per epoch but slightly worse generalization.

6. **3 layers ≈ 4 layers > 5 layers** — diminishing returns from depth. 5 layers with sigmoid literally doesn't learn.

7. **Hidden size 128 > 64 > 32** — wider is better but improvement from 64→128 is smaller than 32→64.

8. **Learning rate 0.001 > 0.0001** — for Adam/NAdam. 0.0001 is too slow for 10 epochs.

9. **Worst configs (< 65%):** SGD + sigmoid + random init — vanishing gradients + poor initialization + no adaptive lr.

10. **10 epochs > 5 epochs** across the board, but some Adam configs already converge by epoch 5.

### Recommendation for ~95% accuracy:
- Adam or NAdam, 3-4 × 128, ReLU, Xavier, lr=0.001, bs=32, wd=0, epochs=15+
- Note: 95% is very hard with a simple feedforward network. Our best was 88.93% test. CNNs would be needed for 95%+.